In [0]:
# -----------------------------------------------------------------------------
# Notebook: 03_ingest_products
#
# Purpose:
#     Ingest products from Azure SQL Database into the Bronze layer.
#
# Source:
#     dbo.Products
#
# Target:
#     Bronze Delta
#
# Load Type:
#     Incremental load (Watermark)
# -----------------------------------------------------------------------------

In [0]:
%run ./../config/00_project_config

In [0]:
%run ./../setup/00_storage_configuration

In [0]:
%run ./../setup/01_sql_configuration

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone

In [0]:
current_ts = datetime.now(timezone.utc)

In [0]:
pipeline_name = "ingest_products_bronze"
bronze_products_path = f"{BRONZE_PATH}/products"

In [0]:
current_watermark = spark.read.format("delta") \
    .load(PIPELINE_WATERMARK_PATH) \
    .filter(F.col("pipeline_name") == pipeline_name) \
    .first()

In [0]:
last_processed_timestamp = datetime(2000,1,1) if current_watermark is None else current_watermark["last_processed_timestamp"]

In [0]:
print(last_processed_timestamp)

In [0]:
sql_query = f"""
SELECT *
FROM dbo.Products
WHERE
    updated_at >= '{last_processed_timestamp}'
"""

In [0]:
df_products_data = spark.read.format("jdbc") \
    .options(
        url = jdbc_url,
        user = jdbc_user,
        password = jdbc_password,
        query = (sql_query)
    ) \
    .load()

In [0]:
df_products_data.printSchema()

In [0]:
display(df_products_data.limit(10))

In [0]:
print(df_products_data.count())

In [0]:
df_is_empty = df_products_data.isEmpty()

if df_is_empty:
    new_last_processed_timestamp = last_processed_timestamp
else:
    new_last_processed_timestamp = df_products_data \
        .agg(F.max("updated_at").alias("updated_at")) \
        .first()["updated_at"]

In [0]:
if not df_is_empty:
    df_products_data.write.format("delta").mode("append").save(bronze_products_path)

In [0]:
df_source_watermark_record = spark.createDataFrame(
    [
        (
            pipeline_name,
            new_last_processed_timestamp,
            current_ts,
            "SUCCESS",
            current_ts,
            current_ts
        )
    ],
    [
        "pipeline_name",
        "last_processed_timestamp",
        "last_run_timestamp",
        "last_run_status",
        "created_at",
        "updated_at"
    ]
)

In [0]:
display(df_source_watermark_record)

In [0]:
from delta.tables import DeltaTable

watermark_table = DeltaTable.forPath(
    spark,
    PIPELINE_WATERMARK_PATH
)

(
    watermark_table.alias("target")
    .merge(
        df_source_watermark_record.alias("source"),
        "target.pipeline_name = source.pipeline_name"
    )
    .whenMatchedUpdate(
        set = {
            "last_processed_timestamp": "source.last_processed_timestamp",
            "last_run_timestamp": "source.last_run_timestamp",
            "last_run_status": "source.last_run_status",
            "updated_at": "source.updated_at"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
display(spark.read.format("delta").load(PIPELINE_WATERMARK_PATH))